In [1]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для поддержки Timestamp из паркета
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [3]:
# Создаем namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS mart")

DataFrame[]

In [5]:
# Создаем таблицу mart.user_trips
spark.sql("""CREATE TABLE mart.user_trips(
  `user_id` BIGINT NOT NULL,
  `trip_id` BIGINT NOT NULL,
  `sex` STRING,
  `age` INT NOT NULL,
  `started_at` DATE,
  `distance` DOUBLE,
  `price` DOUBLE,
  `last_updated` DATE NOT NULL,
  `actuality_date` DATE NOT NULL  
)
USING iceberg""")

DataFrame[]

In [7]:
# Загружаем данные в таблицу
## Для поля age учитываем расчет по месяцам разницы в годах согласно заданию на дату поездки started_at
## Для поля actuality_date используем GREATEST для определения максимальной даты актуальности
## Учитываем данные пользователя актуальные на момент выполнения поездки добавляя условие по дате в JOIN
spark.sql("""INSERT INTO lk.mart.user_trips
  SELECT u.id AS user_id,
    t.id as trip_id,
    u.sex as sex,
    FLOOR(MONTHS_BETWEEN(t.started_at, u.birth_date) / 12) as age,
    t.started_at,
    t.distance,
    t.price,
    current_timestamp() as last_updated,
    GREATEST(u.last_updated,t.last_updated) actuality_date
  FROM dds.users AS u 
    INNER JOIN dds.trips AS t ON u.id=t.user_id
  AND t.started_at between u.start_date and u.end_date
    """)

DataFrame[]